le nom: El jattioui
le prenom: Maryame
Master:GLCC

In [1]:
import numpy as np
import random

# =================================================================
# 1. DÉFINITION DE L'ENVIRONNEMENT (Le "Dataset" Dynamique)
# =================================================================
# On crée un monde de type "GridWorld" (Grille 3x3)
# [0] [1] [2]
# [3] [4] [5: Piège]
# [6] [7] [8: Cible/Sortie]

# Table des récompenses (Rewards) : Chaque case a une valeur
# On pénalise chaque pas (-1) pour forcer l'agent à être efficace.
# On punit lourdement le piège (-10) et on récompense la cible (+10).
rewards = np.array([
    -1, -1, -1,
    -1, -1, -10,
    -1, -1, 10
])

# Actions possibles : 0=Haut, 1=Bas, 2=Gauche, 3=Droite
n_actions = 4
n_states = 9

# =================================================================
# 2. L'ALGORITHME Q-LEARNING (From Scratch)
# =================================================================
class QLearningAgent:
    """
    Agent capable d'apprendre une stratégie optimale dans un environnement
    incertain sans modèle préalable (Model-free RL).
    """
    def __init__(self, states, actions, alpha=0.1, gamma=0.9, epsilon=1.0):
        """
        :param alpha: Learning Rate. Si alpha=0, l'agent n'apprend rien. 
                      Si alpha=1, il ne jure que par la dernière info reçue.
        :param gamma: Discount Factor (Facteur d'actualisation). 
                      Proche de 1 : l'agent est prévoyant (pense au futur).
                      Proche de 0 : l'agent est court-termiste.
        :param epsilon: Taux d'exploration initial.
        """
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = 0.99  # Réduction de l'exploration par épisode
        self.min_epsilon = 0.01    # Seuil minimal d'exploration
        
        # LA Q-TABLE : C'est la mémoire de l'agent.
        # Chaque cellule Q[s, a] représente la 'qualité' d'une action 'a' dans un état 's'.
        # Initialisée à 0 partout : l'agent commence "ignorant".
        self.q_table = np.zeros((states, actions))

    def choose_action(self, state):
        """ 
        MÉCANISME EPSILON-GREEDY :
        C'est l'arbitrage entre l'Exploration (chercher de nouvelles routes)
        et l'Exploitation (utiliser la meilleure route connue).
        """
        if random.uniform(0, 1) < self.epsilon:
            # EXPLORATION : On choisit une action au hasard
            return random.randint(0, n_actions - 1)
        else:
            # EXPLOITATION : On choisit l'action qui a la plus grande valeur Q
            # Si plusieurs actions ont la même valeur, argmax prend la première.
            return np.argmax(self.q_table[state])

    def update_q_table(self, s, a, r, s_next):
        """ 
        MISE À JOUR VIA L'ÉQUATION DE BELLMAN (Temporal Difference)
        Le cœur du Q-Learning : on met à jour la valeur actuelle en fonction 
        de la récompense reçue et de l'estimation du futur.
        """
        # 1. On trouve la meilleure valeur Q possible pour l'état suivant (s_next)
        max_future_q = np.max(self.q_table[s_next])
        
        # 2. Calcul de la cible (Target) : Récompense immédiate + Futur escompté
        target = r + self.gamma * max_future_q
        
        # 3. Calcul de l'erreur temporelle (TD Error)
        # Différence entre ce qu'on vient d'apprendre (target) et ce qu'on croyait (current)
        old_value = self.q_table[s, a]
        
        # 4. Mise à jour pondérée par le learning rate (alpha)
        # Formule : Q_nouveau = Q_ancien + alpha * (Cible - Q_ancien)
        self.q_table[s, a] = old_value + self.alpha * (target - old_value)

    def decay_exploration(self):
        """ Réduit epsilon pour devenir de plus en plus stable/déterministe. """
        if self.epsilon > self.min_epsilon:
            self.epsilon *= self.epsilon_decay

# =================================================================
# 3. MOTEUR DE SIMULATION (Interaction Agent-Environnement)
# =================================================================
def move(state, action):
    """ Calcule la transition d'état dans la grille 3x3. """
    row, col = divmod(state, 3)
    if action == 0 and row > 0: row -= 1    # HAUT
    elif action == 1 and row < 2: row += 1  # BAS
    elif action == 2 and col > 0: col -= 1  # GAUCHE
    elif action == 3 and col < 2: col += 1  # DROITE
    return row * 3 + col

# =================================================================
# 4. PHASE D'ENTRAÎNEMENT (Apprentissage par essais/erreurs)
# =================================================================
agent = QLearningAgent(n_states, n_actions)
episodes = 1000 # Nombre de parties jouées

print(f"Entraînement de l'agent 10_QLearning sur {episodes} épisodes...")

for e in range(episodes):
    current_state = 0 # Point de départ (Haut-Gauche)
    done = False
    
    while not done:
        # L'agent choisit une action (Epsilon-greedy)
        action = agent.choose_action(current_state)
        
        # L'environnement répond avec le nouvel état et la récompense
        next_state = move(current_state, action)
        reward = rewards[next_state]
        
        # L'agent met à jour sa mémoire (Q-Table)
        agent.update_q_table(current_state, action, reward, next_state)
        
        # Transition
        current_state = next_state
        
        # Conditions de fin d'épisode (Victoire ou Défaite)
        if current_state == 8 or current_state == 5:
            done = True
            
    # On réduit le taux d'exploration après chaque épisode
    agent.decay_exploration()

print("Apprentissage terminé.")

# =================================================================
# 5. DÉMONSTRATION DU CHEMIN OPTIMAL (Inférence)
# =================================================================
print("\n--- TEST DU CHEMIN OPTIMAL ---")
state = 0
path = [state]
max_steps = 10

while state != 8 and max_steps > 0:
    # Ici, epsilon est ignoré, on ne fait que de l'EXPLOITATION
    action = np.argmax(agent.q_table[state])
    state = move(state, action)
    path.append(state)
    max_steps -= 1

print(f"Parcours du robot : {path}")
if state == 8:
    print("Résultat : Succès ! L'agent a atteint la sortie en évitant le piège.")
else:
    print("Résultat : Échec ou boucle infinie.")

Entraînement de l'agent 10_QLearning sur 1000 épisodes...
Apprentissage terminé.

--- TEST DU CHEMIN OPTIMAL ---
Parcours du robot : [0, 1, 4, 7, 8]
Résultat : Succès ! L'agent a atteint la sortie en évitant le piège.
